In [ ]:
#| echo: false
%load_ext autoreload
%autoreload 2

# Profile forecasting stages
> Find where MLForecast spends time and memory during fit and cross validation

Pass a `Profiler` through `hooks` to record the nested stages of every call.

In [ ]:
import lightgbm as lgb
from sklearn.linear_model import LinearRegression

from mlforecast import MLForecast
from mlforecast.hooks import Profiler
from mlforecast.utils import generate_daily_series

In [ ]:
series = generate_daily_series(10, min_length=100, max_length=200)

Setting `memory=True` also records the change in the process' resident memory across each stage, which requires psutil.

In [ ]:
profiler = Profiler(memory=True)
fcst = MLForecast(
    models={'lr': LinearRegression(), 'lgb': lgb.LGBMRegressor(verbosity=-1)},
    freq='D',
    lags=[1, 7],
    hooks=[profiler],
)
cv = fcst.cross_validation(series, n_windows=3, h=7)
spans = profiler.to_df()
spans[['stage', 'depth', 'seconds', 'rss_delta_bytes', 'model']]

,stage,depth,seconds,rss_delta_bytes,model
0,cross_validation,0,1.274671,13336576,NaN
1,cv_window,1,0.391066,11083776,NaN
2,fit,2,0.359822,10698752,NaN
3,preprocess,3,0.006702,233472,NaN
4,fit_models,3,0.352794,10465280,NaN
5,fit_model,4,0.002119,921600,lr
6,fit_model,4,0.350266,9543680,lgb
7,predict,2,0.026643,40960,NaN
8,cv_window,1,0.459525,36864,NaN
9,fit,2,0.393929,20480,NaN


To compare model fit costs, select only the `fit_model` rows:

In [ ]:
spans.query("stage == 'fit_model'").groupby('model')['seconds'].sum()

model
lgb    1.048450
lr     0.005085
Name: seconds, dtype: float64

Each stage's duration includes its child stages, so summing every row double counts. Failed stages remain in the table with an `error` value. To profile another run, call `profiler.reset()` after the first run finishes.